# Week 3 * Class 4 -- Subqueries, CTEs & Git
**AI Training Program * Phase 1: Python Foundations**

---

## What you will learn

### SQL (Advanced Fundamentals)
| Concept | Description |
|---|---|
| Scalar Subquery | A nested query that returns exactly one value |
| Row Subquery | A nested query that returns one row, multiple columns |
| Table Subquery | A nested query used as a virtual table |
| CTE (`WITH`) | Named, reusable temporary result sets |
| Set Operations | `UNION`, `INTERSECT`, `EXCEPT` |

### Git & GitHub (Bootcamp Style)
| Lab | What you will do |
|---|---|
| **Lab 1** | Configure Git identity |
| **Lab 2** | Create a GitHub repo via the UI |
| **Lab 3** | Clone, add files, commit, push |
| **Lab 4** | Branching & merging |
| **Lab 5** | Pull Requests |
| **Lab 6** | Resolving merge conflicts |
| **Lab 7** | Rebase workflow |


---
## Database Setup

We use **SQLite** (an in-process database -- no server needed) with three related tables.

```
students ---+
            +--- enrollments --- courses
```

**Why these tables?** They model a real-world many-to-many relationship:
one student can enroll in many courses; one course can have many students.


In [ ]:
import sqlite3
import pandas as pd

conn   = sqlite3.connect("school.db")
cursor = conn.cursor()

cursor.executescript(
    """
    DROP TABLE IF EXISTS enrollments;
    DROP TABLE IF EXISTS students;
    DROP TABLE IF EXISTS courses;

    -- students: each student belongs to a dept and has a GPA
    CREATE TABLE students (
        student_id INTEGER PRIMARY KEY AUTOINCREMENT,
        name       TEXT NOT NULL,
        city       TEXT,
        gpa        REAL,
        dept       TEXT
    );

    -- courses: each course has a dept and credit hours
    CREATE TABLE courses (
        course_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        title      TEXT NOT NULL,
        dept       TEXT,
        credits    INTEGER DEFAULT 3
    );

    -- enrollments: join table, one row per (student, course) pair
    -- score: the student score in that course; status: active or dropped
    CREATE TABLE enrollments (
        enroll_id  INTEGER PRIMARY KEY AUTOINCREMENT,
        student_id INTEGER REFERENCES students(student_id),
        course_id  INTEGER REFERENCES courses(course_id),
        score      REAL,
        status     TEXT DEFAULT 'active'
    );

    INSERT INTO students (name, city, gpa, dept) VALUES
        ('Aarav Shah',   'Mumbai',    3.8, 'AI'),
        ('Priya Nair',   'Pune',      3.5, 'AI'),
        ('Rohan Mehta',  'Delhi',     2.9, 'CS'),
        ('Sana Khan',    'Mumbai',    3.7, 'AI'),
        ('Dev Sharma',   'Bangalore', 3.1, 'CS'),
        ('Anita Bose',   'Kolkata',   3.9, 'AI'),
        ('Kiran Rao',    'Hyderabad', NULL,'CS'),
        ('Meera Pillai', 'Mumbai',    3.6, 'CS'),
        ('Arjun Gupta',  'Delhi',     2.7, 'Math'),
        ('Sneha Iyer',   'Chennai',   3.4, 'Math');

    INSERT INTO courses (title, dept, credits) VALUES
        ('Python Foundations', 'CS',   4),
        ('Machine Learning',   'AI',   4),
        ('SQL & Databases',    'CS',   3),
        ('Deep Learning',      'AI',   4),
        ('MLOps',              'AI',   3),
        ('Data Visualisation', 'Math', 3);

    INSERT INTO enrollments (student_id, course_id, score, status) VALUES
        (1, 1, 88, 'active'),  (1, 2, 75, 'active'),  (1, 3, 91, 'active'),
        (2, 1, 72, 'active'),  (2, 4, 84, 'active'),
        (3, 2, 61, 'active'),  (3, 3, 55, 'dropped'),
        (4, 1, 95, 'active'),  (4, 2, 88, 'active'),  (4, 5, 79, 'active'),
        (5, 3, 70, 'active'),  (5, 4, 66, 'active'),
        (6, 1, 98, 'active'),  (6, 2, 92, 'active'),  (6, 4, 89, 'active'),
        (7, 5, 77, 'active'),
        (8, 1, 83, 'active'),  (8, 3, 76, 'active'),
        (9, 2, 58, 'dropped'), (9, 6, 64, 'active'),
        (10,1, 81, 'active'),  (10,6, 73, 'active');
    """
)
conn.commit()
print("Database ready -- 10 students, 6 courses, 21 enrollments")


---
## Part A: Subqueries

### What is a Subquery?

A **subquery** (also called an *inner query* or *nested query*) is a `SELECT` statement written **inside another SQL statement**. The database executes the inner query first, then uses its result in the outer query.

```sql
SELECT column
FROM   table
WHERE  column > (SELECT AVG(column) FROM table);   -- this is the subquery
```

### Three Types of Subqueries

| Type | Returns | Typical Location | Performance |
|---|---|---|---|
| **Scalar** | Exactly 1 value (1 row x 1 col) | `WHERE`, `SELECT`, `HAVING` | Fast |
| **Row** | Exactly 1 row, multiple columns | `WHERE` with row comparison | Fast |
| **Table** (derived table) | Multiple rows & columns | `FROM`, `JOIN`, `IN` | Moderate |

> **Rule:** If a subquery returns more than one value where a scalar is expected, SQL raises an error.
> Use `IN` when the subquery may return multiple rows.

### Mental Model

Think of a subquery as answering a *pre-question* before the main query:

> *Give me all students whose GPA is above average.*
> Pre-question -> *What IS the average GPA?* -> subquery answers this
> Main query -> uses that answer to filter


### A.1 -- Scalar Subquery

A **scalar subquery** returns exactly **one row and one column** -- a single value.
It can appear anywhere a single literal value is expected.

**Syntax:**
```sql
WHERE column > (SELECT single_aggregate FROM table)
```

**How it executes:**
1. Database runs `SELECT AVG(gpa) FROM students` -> gets e.g. `3.46`
2. Database substitutes: `WHERE gpa > 3.46`
3. Returns matching rows

> Warning: If the inner query returns more than one row, SQL raises `subquery returns more than one row`.


In [ ]:
# A.1a: Scalar in WHERE clause
# Find students whose GPA is above the class average.
# The subquery (SELECT AVG(gpa) ...) runs first, outer WHERE uses the result.

pd.read_sql_query(
    """
    SELECT
        name,
        gpa,
        ROUND((SELECT AVG(gpa) FROM students WHERE gpa IS NOT NULL), 2) AS class_avg,
        ROUND(gpa - (SELECT AVG(gpa) FROM students WHERE gpa IS NOT NULL), 2) AS above_avg_by
    FROM   students
    WHERE  gpa > (SELECT AVG(gpa) FROM students)
    ORDER BY gpa DESC
    """,
    conn
)


In [ ]:
# A.1b: Scalar in SELECT clause
# The scalar subquery adds a computed column to every row.
# Same value for each row, useful for comparison.

pd.read_sql_query(
    """
    SELECT
        name,
        dept,
        gpa,
        ROUND((SELECT AVG(gpa) FROM students WHERE gpa IS NOT NULL), 2) AS class_avg
    FROM   students
    WHERE  gpa IS NOT NULL
    ORDER BY gpa DESC
    """,
    conn
)


### A.2 -- Table Subquery with `IN` / `NOT IN`

A **table subquery** returns multiple rows. When used with `IN`, the outer query checks
whether a value exists in that result set.

**Syntax:**
```sql
WHERE column IN (SELECT column FROM table WHERE condition)
```

**How it executes:**
1. Inner query runs -> produces a list of values, e.g. `[1, 2, 4, 6]`
2. Outer query filters: `WHERE student_id IN (1, 2, 4, 6)`

> `NOT IN` is the opposite -- it excludes rows whose value appears in the subquery result.
> **Gotcha:** If the subquery result contains `NULL`, `NOT IN` returns nothing.
> Always add `WHERE column IS NOT NULL` inside the subquery when using `NOT IN`.


In [ ]:
# A.2a: IN -- which students are actively enrolled in Machine Learning?
# Inner query: find student_ids enrolled in ML
# Outer query: look up their names

pd.read_sql_query(
    """
    SELECT name, dept, gpa
    FROM   students
    WHERE  student_id IN (
        SELECT e.student_id
        FROM   enrollments e
        JOIN   courses     c ON e.course_id = c.course_id
        WHERE  c.title  = 'Machine Learning'
          AND  e.status = 'active'
    )
    ORDER BY gpa DESC
    """,
    conn
)


In [ ]:
# A.2b: NOT IN -- find students who have NOT enrolled in any AI-dept course
# DISTINCT in subquery avoids duplicate student_ids

pd.read_sql_query(
    """
    SELECT name, dept, city
    FROM   students
    WHERE  student_id NOT IN (
        SELECT DISTINCT e.student_id
        FROM   enrollments e
        JOIN   courses     c ON e.course_id = c.course_id
        WHERE  c.dept = 'AI'
          AND  e.student_id IS NOT NULL   -- safety: exclude NULLs for NOT IN
    )
    ORDER BY name
    """,
    conn
)


### A.3 -- Subquery in `FROM` (Derived Table)

When a subquery appears in the `FROM` clause, it acts as a **temporary virtual table**
(also called a *derived table* or *inline view*). You must give it an alias.

**Syntax:**
```sql
SELECT ...
FROM   (SELECT ... FROM table GROUP BY ...) AS alias
JOIN   other_table ON alias.col = other_table.col
```

**Why use it?** When you need to aggregate data first, then filter or join on the result --
something you cannot do in a single `WHERE` clause.

> Modern SQL prefers CTEs (`WITH`) over derived tables for readability (see Part B).
> But derived tables still appear in legacy code and are important to recognize.


In [ ]:
# A.3: Derived Table (subquery in FROM)
# Goal: find the top-scoring student per course.
# Step 1 (inner): GROUP enrollments -> max score per course
# Step 2 (outer): JOIN that result with courses to get course titles

pd.read_sql_query(
    """
    SELECT
        c.title        AS course,
        c.dept,
        sub.student_id,
        sub.max_score
    FROM   courses c
    JOIN (
        SELECT
            course_id,
            student_id,
            MAX(score) AS max_score
        FROM   enrollments
        WHERE  status = 'active'
        GROUP BY course_id
    ) AS sub ON c.course_id = sub.course_id
    ORDER BY max_score DESC
    """,
    conn
)


---
## Part B: Common Table Expressions (CTEs)

### What is a CTE?

A **CTE** (Common Table Expression) is a named, temporary result set defined at the
top of a query using the `WITH` keyword. Think of it as giving a subquery a name
and moving it out of the main query for clarity.

**Syntax:**
```sql
WITH cte_name AS (
    SELECT ...          -- the CTE definition
    FROM   ...
    WHERE  ...
)
SELECT ...              -- the main query references the CTE by name
FROM   cte_name
WHERE  ...;
```

### Subquery vs CTE -- What is the Difference?

| Feature | Subquery | CTE |
|---|---|---|
| Location | Inline (inside the query) | Before the main query (`WITH`) |
| Reusability | Cannot reference it twice | Can reference it multiple times |
| Readability | Gets messy with nesting | Clean -- reads top to bottom |
| Debugging | Hard to test in isolation | Easy -- can `SELECT * FROM cte` to check |
| Performance | Same as CTE in most engines | Same as subquery in most engines |

> **Rule:** Prefer CTEs over derived tables. They make code **readable, maintainable, and debuggable**.

### Chaining Multiple CTEs

```sql
WITH
cte1 AS (SELECT ... FROM ...),        -- first CTE
cte2 AS (SELECT ... FROM cte1 ...),   -- can reference cte1!
cte3 AS (SELECT ... FROM cte2 ...)    -- can reference cte1, cte2
SELECT ... FROM cte3;                  -- main query
```


In [ ]:
# B.1: Basic CTE -- same as the derived-table example, but readable
# The CTE student_scores is defined once and referenced in the main SELECT.

pd.read_sql_query(
    """
    WITH student_scores AS (
        SELECT
            s.student_id,
            s.name,
            s.dept,
            COUNT(e.course_id)     AS courses_taken,
            ROUND(AVG(e.score), 1) AS avg_score,
            MAX(e.score)           AS best_score,
            MIN(e.score)           AS worst_score
        FROM   students    s
        JOIN   enrollments e ON s.student_id = e.student_id
        WHERE  e.status = 'active'
        GROUP BY s.student_id, s.name, s.dept
    )
    SELECT *
    FROM   student_scores
    WHERE  avg_score > 75
    ORDER BY avg_score DESC
    """,
    conn
)


In [ ]:
# B.2: Chained CTEs -- build analysis in named steps

pd.read_sql_query(
    """
    WITH
    -- CTE 1: compute average score per course (active only)
    course_avg AS (
        SELECT
            course_id,
            COUNT(*)               AS enrolled,
            ROUND(AVG(score), 1)   AS avg_score
        FROM   enrollments
        WHERE  status = 'active'
        GROUP BY course_id
    ),

    -- CTE 2: join with courses table and classify each course
    course_tier AS (
        SELECT
            c.title,
            c.dept,
            ca.enrolled,
            ca.avg_score,
            CASE
                WHEN ca.avg_score >= 85 THEN 'High Performer'
                WHEN ca.avg_score >= 70 THEN 'On Track'
                ELSE                         'Needs Attention'
            END AS tier
        FROM   courses    c
        JOIN   course_avg ca ON c.course_id = ca.course_id
    ),

    -- CTE 3: summarise tiers
    tier_summary AS (
        SELECT
            tier,
            COUNT(*)                   AS num_courses,
            ROUND(AVG(avg_score), 1)   AS avg_of_avgs
        FROM   course_tier
        GROUP BY tier
    )

    SELECT * FROM tier_summary
    ORDER BY avg_of_avgs DESC
    """,
    conn
)


---
## Part D: Set Operations -- `UNION`, `INTERSECT`, `EXCEPT`

### What are Set Operations?

Set operations combine the results of **two or more SELECT statements** into a single result set.

**Rules (same for all three):**
1. Both queries must return the **same number of columns**
2. Corresponding columns must have **compatible data types**
3. Column names in the result come from the **first query**

### Visual Guide

```
Query A result:  [1, 2, 3, 4]
Query B result:  [3, 4, 5, 6]

UNION:           [1, 2, 3, 4, 5, 6]         all unique values from A + B
UNION ALL:       [1, 2, 3, 4, 3, 4, 5, 6]   all values, duplicates kept
INTERSECT:       [3, 4]                      only values in BOTH A and B
EXCEPT:          [1, 2]                      values in A but NOT in B
```

| Operation | Returns | Duplicates |
|---|---|---|
| `UNION` | All rows from A + B | Removed |
| `UNION ALL` | All rows from A + B | Kept (faster!) |
| `INTERSECT` | Only rows in both A AND B | Removed |
| `EXCEPT` | Rows in A but NOT in B | Removed |


In [ ]:
# D.1: UNION -- combine AI students and AI courses into one list

pd.read_sql_query(
    """
    SELECT name AS label, 'student' AS source FROM students WHERE dept = 'AI'
    UNION
    SELECT title AS label, 'course'  AS source FROM courses  WHERE dept = 'AI'
    ORDER BY source, label
    """,
    conn
)


In [ ]:
# D.2: INTERSECT -- department names that exist in BOTH students AND courses

pd.read_sql_query(
    """
    SELECT DISTINCT dept FROM students WHERE dept IS NOT NULL
    INTERSECT
    SELECT DISTINCT dept FROM courses
    """,
    conn
)


In [ ]:
# D.3: EXCEPT -- student depts that do NOT appear as a course dept

pd.read_sql_query(
    """
    SELECT DISTINCT dept FROM students WHERE dept IS NOT NULL
    EXCEPT
    SELECT DISTINCT dept FROM courses
    """,
    conn
)


In [ ]:
# D.4: Practical UNION -- all high-GPA students OR top scorers

pd.read_sql_query(
    """
    SELECT s.name, s.gpa AS metric, 'High GPA' AS reason
    FROM   students s
    WHERE  s.gpa >= 3.7

    UNION

    SELECT s.name, MAX(e.score) AS metric, 'Top Score (90+)' AS reason
    FROM   students    s
    JOIN   enrollments e ON s.student_id = e.student_id
    WHERE  e.score >= 90 AND e.status = 'active'
    GROUP BY s.name

    ORDER BY metric DESC
    """,
    conn
)


---
## Part F: Git & GitHub -- Bootcamp Style

### Core Concepts

| Term | What it is |
|---|---|
| **Repository (repo)** | A folder tracked by Git; contains your project + full history |
| **Commit** | A snapshot of the repo at a point in time; has a unique hash |
| **Branch** | A parallel version of the repo; `main` is the default |
| **Remote** | A copy of the repo hosted elsewhere (GitHub, GitLab, etc.) |
| **Push** | Send local commits to the remote |
| **Pull** | Download remote commits to local |
| **Merge** | Combine changes from two branches |
| **Rebase** | Replay commits on top of another branch (rewrites history) |

### The Four-Zone Model

```
Working Tree   --git add-->  Staging Area  --git commit-->  Local Repo  --git push-->  GitHub
(edited files)               (index)                       (.git folder)              (remote)
     ^                                                            |
     +----------------------- git checkout / git restore ---------+
```


### Lab 1 -- Configure Git Identity

**Why?** Every commit records who made it. This is a global config -- done once per machine.


In [ ]:
# Lab 1: First-time Git configuration
# Fill in your real name and email (should match your GitHub account)

# !git config --global user.name  "Your Full Name"
# !git config --global user.email "you@example.com"
# !git config --global init.defaultBranch main      # use main as default branch
# !git config --global core.editor "code --wait"    # use VS Code as editor

import subprocess
result = subprocess.run(["git", "config", "--list", "--global"], capture_output=True, text=True)
lines = [l for l in result.stdout.splitlines() if any(k in l for k in ['user.', 'init.', 'core.editor'])]
print("Current Git config:")
for line in lines:
    print(" ", line)
if not lines:
    print("  (not yet configured -- uncomment the lines above and run again)")


### Lab 2 -- Create a GitHub Repository via the UI

Do this in your browser:

#### Step 1: Log in to GitHub
Go to https://github.com and sign in.

#### Step 2: Create a New Repository
1. Click the **+** icon in the top-right corner -> **New repository**
2. Fill in the form:

| Field | Value |
|---|---|
| **Repository name** | `ai-training-week3` |
| **Description** | `Week 3: SQL subqueries, CTEs, and Git` |
| **Visibility** | Public (or Private) |
| **Initialize with README** | Leave UNCHECKED |
| **Add .gitignore** | Leave UNCHECKED |
| **License** | None |

3. Click **Create repository**

#### Step 3: Copy the Remote URL
After creating, copy the HTTPS URL -- it looks like:
```
https://github.com/YOUR_USERNAME/ai-training-week3.git
```

> **HTTPS vs SSH:** HTTPS uses your username + personal access token.
> SSH uses a key pair -- more secure once set up. For this lab, use HTTPS.

**Checkpoint:** You have a blank repo on GitHub. Next lab: connect your local code to it.


### Lab 3 -- Init, Add, Commit, Push

**Goal:** Create a local project, take your first snapshot, and push it to GitHub.


In [ ]:
# Lab 3a: Initialise a new local repository
import os, subprocess

demo_dir = "/tmp/ai-training-week3"
os.makedirs(demo_dir, exist_ok=True)
os.chdir(demo_dir)

result = subprocess.run(["git", "init"], capture_output=True, text=True)
print(result.stdout.strip())

result = subprocess.run(["git", "status"], capture_output=True, text=True)
print(result.stdout.strip())


In [ ]:
# Lab 3b: Create a .gitignore
# .gitignore tells Git which files/folders to NEVER track.

gitignore = """# Python
__pycache__/
*.py[cod]
*.egg-info/

# Virtual environments
.env
venv/
.venv/

# Jupyter
.ipynb_checkpoints/

# Data files
*.csv
*.db
*.sqlite

# Secrets -- NEVER commit these
.env.local
secrets.yaml
*.pem
*.key

# OS generated
.DS_Store
Thumbs.db
"""

with open(".gitignore", "w") as f:
    f.write(gitignore)

print(".gitignore written")


In [ ]:
# Lab 3c: Create project files
with open("README.md", "w") as f:
    f.write("# AI Training Program -- Week 3\n\n")
    f.write("SQL subqueries, CTEs, Window Functions, Set Operations, Indexes.\n")

with open("main.py", "w") as f:
    f.write("# Week 3 Project -- SQL Query Functions\n")
    f.write("import sqlite3\n")
    f.write("import pandas as pd\n\n")
    f.write("def get_top_students(db_path, n=5):\n")
    f.write("    conn = sqlite3.connect(db_path)\n")
    f.write("    df = pd.read_sql_query(\n")
    f.write("        'SELECT name, gpa FROM students WHERE gpa IS NOT NULL ORDER BY gpa DESC LIMIT ?',\n")
    f.write("        conn, params=[n])\n")
    f.write("    conn.close()\n")
    f.write("    return df\n")

import subprocess
r = subprocess.run(["ls", "-la"], capture_output=True, text=True)
print("Files created:")
print(r.stdout)


In [ ]:
# Lab 3d: Stage files (git add)
# git add moves files from the Working Tree to the Staging Area.

import subprocess

print("=== BEFORE git add ===")
r = subprocess.run(["git", "status"], capture_output=True, text=True)
print(r.stdout)

subprocess.run(["git", "add", ".gitignore", "README.md", "main.py"])

print("=== AFTER git add ===")
r = subprocess.run(["git", "status"], capture_output=True, text=True)
print(r.stdout)


In [ ]:
# Lab 3e: First commit
# Conventional Commits format (industry standard):
#   <type>(<scope>): <description>
#   Types: feat | fix | docs | style | refactor | test | chore | perf

import subprocess

subprocess.run(["git", "config", "user.email", "student@example.com"])
subprocess.run(["git", "config", "user.name",  "AI Student"])

r = subprocess.run(
    ["git", "commit", "-m", "feat: initial project setup with SQL query functions"],
    capture_output=True, text=True
)
print(r.stdout or r.stderr)

print("\n=== git log ===")
r = subprocess.run(["git", "log", "--oneline", "--graph", "--decorate"], capture_output=True, text=True)
print(r.stdout)


In [ ]:
# Lab 3f: Make a change and commit again
# Edit a file -> stage -> commit. This is the everyday Git loop.

with open("main.py", "a") as f:
    f.write("\n# Added in Week 3 Class 4\n")

import subprocess
subprocess.run(["git", "add", "main.py"])
r = subprocess.run(
    ["git", "commit", "-m", "docs: add class reference comment"],
    capture_output=True, text=True
)
print(r.stdout)

print("\n=== Updated log ===")
r = subprocess.run(["git", "log", "--oneline", "--decorate"], capture_output=True, text=True)
print(r.stdout)


### Lab 3g -- Push to GitHub (run in terminal)

After creating the repo in Lab 2, run these in your terminal:

```bash
# Tell Git where GitHub is (replace YOUR_USERNAME)
git remote add origin https://github.com/YOUR_USERNAME/ai-training-week3.git

# Rename branch to main if needed
git branch -M main

# Push and set upstream tracking (-u means future git push works without args)
git push -u origin main
```

**What each flag does:**
- `remote add origin` -- registers the GitHub URL under the name `origin` (convention)
- `-M main` -- rename branch to `main`
- `-u origin main` -- push + set upstream; after this, just `git push` works

**Verify on GitHub:** Refresh your GitHub repo page -- you should see your files!

> If you get a 403 error, GitHub requires a **Personal Access Token** instead of a password.
> Go to: GitHub -> Settings -> Developer settings -> Personal access tokens -> Generate new token -> check `repo` scope.


### Lab 4 -- Branching & Merging

**Why branches?**
- `main` is your stable, production-ready code
- You develop new features on **feature branches** -- isolated experiments
- When ready, you merge the branch back into `main`

```
main:    A --- B ----------------------- M (merge commit)
                  \                     /
feature:           C --- D ----------- /
```

**Branch naming conventions:**
- `feature/add-user-auth` -- new functionality
- `fix/null-pointer-crash` -- bug fix
- `docs/update-readme` -- documentation only
- `chore/update-deps` -- maintenance


In [ ]:
# Lab 4a: Create and switch to a feature branch
import subprocess, os
os.chdir("/tmp/ai-training-week3")

# git checkout -b name = create branch AND switch to it in one command
r = subprocess.run(["git", "checkout", "-b", "feature/db-utils"], capture_output=True, text=True)
print(r.stdout or r.stderr)

r = subprocess.run(["git", "branch"], capture_output=True, text=True)
print("Branches (* = current):")
print(r.stdout)


In [ ]:
# Lab 4b: Add feature code on the branch
with open("db_utils.py", "w") as f:
    f.write("# Database utility helpers\n")
    f.write("import sqlite3\n\n")
    f.write("def connect(db_path='school.db'):\n")
    f.write("    '''Return a SQLite connection with foreign keys enabled.'''\n")
    f.write("    conn = sqlite3.connect(db_path)\n")
    f.write("    conn.execute('PRAGMA foreign_keys = ON')\n")
    f.write("    return conn\n")

import subprocess
subprocess.run(["git", "add", "db_utils.py"])
r = subprocess.run(
    ["git", "commit", "-m", "feat(db): add db_utils with connection helper"],
    capture_output=True, text=True
)
print(r.stdout)

print("\n=== Branch log ===")
r = subprocess.run(["git", "log", "--oneline", "--graph", "--all", "--decorate"], capture_output=True, text=True)
print(r.stdout)


In [ ]:
# Lab 4c: Merge the feature branch into main
import subprocess

subprocess.run(["git", "checkout", "main"])

# --no-ff forces a merge commit even if fast-forward is possible.
# This preserves the branch history in the graph.
r = subprocess.run(
    ["git", "merge", "feature/db-utils", "--no-ff", "-m", "merge: feature/db-utils into main"],
    capture_output=True, text=True
)
print(r.stdout or r.stderr)

print("\n=== Graph after merge ===")
r = subprocess.run(
    ["git", "log", "--oneline", "--graph", "--all", "--decorate"],
    capture_output=True, text=True
)
print(r.stdout)


In [ ]:
# Lab 4d: Clean up merged branch
import subprocess
r = subprocess.run(["git", "branch", "-d", "feature/db-utils"], capture_output=True, text=True)
print(r.stdout or r.stderr)

print("Remaining branches:")
r = subprocess.run(["git", "branch"], capture_output=True, text=True)
print(r.stdout)


### Lab 5 -- Pull Requests (GitHub UI)

A **Pull Request (PR)** is GitHub way to:
1. Propose merging a branch into another
2. Ask for code review from teammates
3. Discuss changes before they land in `main`

**Workflow (do this in your browser after pushing a feature branch):**

#### Step 1: Push your feature branch
```bash
git push -u origin feature/reporting
```

#### Step 2: Open a Pull Request on GitHub
1. Go to your repo on GitHub
2. Click the **Compare & pull request** button
3. Fill in the form:

| Field | What to write |
|---|---|
| **Title** | `feat: add reporting module` |
| **Description** | What changed, why, how to test |
| **Base branch** | `main` (the target) |
| **Compare branch** | `feature/reporting` (your changes) |

4. Click **Create pull request**

#### Step 3: Review
- Click **Files changed** tab -- see a diff of every change
- Add inline comments on specific lines
- Click **Review changes** -> Approve

#### Step 4: Merge the PR
- Click **Merge pull request** -> **Confirm merge**
- Delete the branch (GitHub offers this automatically)

#### Step 5: Sync local main
```bash
git checkout main
git pull origin main
```

> **Draft PRs:** Use *Create draft pull request* for work-in-progress.
> **Protected branches:** In team settings, require PR approvals before merging to `main`.


### Lab 6 -- Resolving Merge Conflicts

A **merge conflict** occurs when two branches modify the **same lines** of the same file.
Git does not know which change to keep -- it asks you to decide.

**Conflict markers look like this:**
```
<<<<<<< HEAD                  <- your current branch version
def greet():
    print('Hello, World!')
=======                       <- separator
def greet():
    print('Hi there!')
>>>>>>> feature/greeting      <- the branch being merged in
```

**Resolution steps:**
1. Edit the file: remove the markers, keep what you want
2. `git add <file>` -- mark as resolved
3. `git commit` -- complete the merge


In [ ]:
# Lab 6a: Simulate a merge conflict
import subprocess, os
os.chdir("/tmp/ai-training-week3")

# Branch A: edit README one way
subprocess.run(["git", "checkout", "-b", "branch-a"])
with open("README.md", "w") as f:
    f.write("# AI Training Program\nVersion: 1.0 -- developed by Team Alpha\n")
subprocess.run(["git", "add", "README.md"])
subprocess.run(["git", "commit", "-m", "docs: set version 1.0 from branch-a"])

# Switch to main and create Branch B with a conflicting change
subprocess.run(["git", "checkout", "main"])
subprocess.run(["git", "checkout", "-b", "branch-b"])
with open("README.md", "w") as f:
    f.write("# AI Training Program\nVersion: 2.0 -- developed by Team Beta\n")
subprocess.run(["git", "add", "README.md"])
subprocess.run(["git", "commit", "-m", "docs: set version 2.0 from branch-b"])

# Merge branch-a into branch-b -- CONFLICT!
print("=== Attempting merge (expect conflict) ===")
r = subprocess.run(["git", "merge", "branch-a"], capture_output=True, text=True)
print(r.stdout or r.stderr)

print("\n=== Conflicted README.md ===")
print(open("README.md").read())


In [ ]:
# Lab 6b: Resolve the conflict
import subprocess

# Resolution: keep the best of both
with open("README.md", "w") as f:
    f.write("# AI Training Program\n")
    f.write("Version: 2.0 -- developed by Team Alpha & Beta\n\n")
    f.write("## Overview\n")
    f.write("SQL subqueries, CTEs, Window Functions, Set Operations, Indexes.\n")

# Mark as resolved and complete the merge
subprocess.run(["git", "add", "README.md"])
r = subprocess.run(
    ["git", "commit", "-m", "merge: resolve version conflict -- combine team credits"],
    capture_output=True, text=True
)
print(r.stdout or r.stderr)

print("\n=== Resolved README ===")
print(open("README.md").read())

print("\n=== Log after conflict resolution ===")
r = subprocess.run(["git", "log", "--oneline", "--graph", "--all", "--decorate"], capture_output=True, text=True)
print(r.stdout)

# Cleanup
subprocess.run(["git", "checkout", "main"])
subprocess.run(["git", "branch", "-D", "branch-a"])
subprocess.run(["git", "branch", "-D", "branch-b"])


### Lab 7 -- Rebase

### Merge vs Rebase -- Key Difference

Both `merge` and `rebase` integrate changes from one branch into another.
The difference is **how they do it** and **what history looks like afterward**.

#### Merge
```
main:    A --- B ---------------------- M (merge commit)
                  \                   /
feature:           C --- D --------- /
```
- Creates a **merge commit** with two parents
- Preserves full history -- you can see exactly when branches diverged

#### Rebase
```
main:    A --- B --- C' --- D'    (C and D are replayed as new commits)
```
- **Replays** your feature commits on top of the latest `main`
- Creates a **linear history** -- as if you wrote the feature after all of main
- Cleaner `git log` but rewrites commit hashes

### Golden Rule of Rebase

> **NEVER rebase commits that have been pushed to a shared remote branch.**
> Rebase rewrites history -- anyone else who has your old commits will have a diverged history.
> Rebase only LOCAL branches before pushing.

### When to Use Each

| Scenario | Use |
|---|---|
| Feature branch -> main in team setting | **Merge** (preserves context) |
| Long-lived feature, want to keep up with main | **Rebase** (local only) |
| Cleaning up messy local commits before PR | **Interactive Rebase** |
| Shared branch (multiple people pushing) | **Merge only** |


In [ ]:
# Lab 7a: Set up a rebase scenario
import subprocess, os
os.chdir("/tmp/ai-training-week3")

# Start a feature branch from current main
subprocess.run(["git", "checkout", "-b", "feature/stats-module"])

# Add a commit on the feature branch
with open("stats.py", "w") as f:
    f.write("def mean(values):\n    return sum(values) / len(values)\n")
subprocess.run(["git", "add", "stats.py"])
subprocess.run(["git", "commit", "-m", "feat(stats): add mean function"])

# Meanwhile, main gets a new commit (simulating teammate work)
subprocess.run(["git", "checkout", "main"])
with open("CHANGELOG.md", "w") as f:
    f.write("# Changelog\n\n## v1.0 -- Initial SQL & Git project\n")
subprocess.run(["git", "add", "CHANGELOG.md"])
subprocess.run(["git", "commit", "-m", "docs: add CHANGELOG"])

print("=== History BEFORE rebase ===")
r = subprocess.run(
    ["git", "log", "--oneline", "--graph", "--all", "--decorate"],
    capture_output=True, text=True
)
print(r.stdout)


In [ ]:
# Lab 7b: Perform the rebase
import subprocess

# Switch to feature branch and rebase onto main
subprocess.run(["git", "checkout", "feature/stats-module"])

r = subprocess.run(["git", "rebase", "main"], capture_output=True, text=True)
print("Rebase output:", r.stdout or r.stderr)

print("\n=== History AFTER rebase ===")
r = subprocess.run(
    ["git", "log", "--oneline", "--graph", "--all", "--decorate"],
    capture_output=True, text=True
)
print(r.stdout)
# Notice: feature branch commits now appear AFTER main latest commit
# History is linear -- no merge commit


In [ ]:
# Lab 7c: Interactive rebase -- squash messy commits
# In real use: git rebase -i HEAD~n opens an editor where you pick:
#   squash  -> combine commits
#   reword  -> edit commit message
#   drop    -> delete a commit

import subprocess

# Simulate 3 messy commits
for i, msg in enumerate(["WIP", "fix typo", "fix typo again"], start=1):
    with open("stats.py", "a") as f:
        f.write(f"# note {i}\n")
    subprocess.run(["git", "add", "stats.py"])
    subprocess.run(["git", "commit", "-m", msg])

print("=== Before squash (messy history) ===")
r = subprocess.run(["git", "log", "--oneline", "main..HEAD"], capture_output=True, text=True)
print(r.stdout)

# Squash last 3 commits into one using reset --soft
subprocess.run(["git", "reset", "--soft", "HEAD~3"])
subprocess.run(["git", "add", "."])
r = subprocess.run(
    ["git", "commit", "-m", "feat(stats): add mean function (cleaned up)"],
    capture_output=True, text=True
)
print("\n=== After squash (clean history) ===")
r = subprocess.run(["git", "log", "--oneline", "main..HEAD"], capture_output=True, text=True)
print(r.stdout)


In [ ]:
# Lab 7d: Fast-forward merge after rebase
# After rebasing, the merge into main is a clean fast-forward (no merge commit needed)

import subprocess
subprocess.run(["git", "checkout", "main"])
r = subprocess.run(["git", "merge", "feature/stats-module"], capture_output=True, text=True)
print(r.stdout or r.stderr)

print("\n=== Final linear history ===")
r = subprocess.run(
    ["git", "log", "--oneline", "--graph", "--decorate"],
    capture_output=True, text=True
)
print(r.stdout)

subprocess.run(["git", "branch", "-d", "feature/stats-module"])


### Complete Git Command Reference

#### Daily Workflow
| Command | What it does |
|---|---|
| `git status` | Show working tree status |
| `git add <file>` | Stage a specific file |
| `git add .` | Stage ALL changes |
| `git add -p` | Interactively stage chunks |
| `git commit -m "msg"` | Commit staged changes |
| `git commit --amend` | Edit the last commit |
| `git log --oneline --graph --all` | Visual branch history |
| `git diff` | Show unstaged changes |
| `git diff --staged` | Show staged changes |

#### Branching
| Command | What it does |
|---|---|
| `git branch` | List all branches |
| `git checkout -b name` | Create + switch to new branch |
| `git switch -c name` | Modern equivalent of checkout -b |
| `git branch -d name` | Delete a branch (safe) |
| `git branch -D name` | Force delete a branch |
| `git merge branch` | Merge branch into current branch |
| `git rebase main` | Replay current branch commits on top of main |
| `git rebase -i HEAD~n` | Interactive rebase last n commits |

#### Remote (GitHub)
| Command | What it does |
|---|---|
| `git remote add origin <url>` | Register a remote |
| `git push -u origin main` | Push and set upstream |
| `git push` | Push current branch to upstream |
| `git pull` | Fetch + merge from upstream |
| `git fetch` | Download remote changes without merging |
| `git clone <url>` | Download a repo |

#### Undoing Things
| Command | What it does |
|---|---|
| `git restore <file>` | Discard working tree changes |
| `git reset HEAD <file>` | Unstage a file |
| `git reset --soft HEAD~1` | Undo last commit, keep staged |
| `git reset --hard HEAD~1` | Undo last commit, discard changes (WARNING) |
| `git revert <hash>` | Create a new commit that undoes a previous one |
| `git stash` | Temporarily save uncommitted changes |
| `git stash pop` | Restore stashed changes |


---
## Exercises

### SQL Exercises


**Exercise 1 (CTE + Subquery):** Write a CTE that calculates each student average score across active enrollments. Then, in the main query, select only those students whose average score is above the overall average score of all students.

*Hint: The overall average is a scalar subquery in the `WHERE` clause of the main query.*


In [ ]:
# Exercise 1 -- your solution here
pd.read_sql_query(
    """
    WITH student_avg AS (
        SELECT s.name, ROUND(AVG(e.score), 1) AS avg_score
        FROM   students s
        JOIN   enrollments e ON s.student_id = e.student_id
        WHERE  e.status = 'active'
        GROUP BY s.name
    )
    SELECT *
    FROM   student_avg
    WHERE  avg_score > (SELECT AVG(avg_score) FROM student_avg)
    ORDER BY avg_score DESC
    """,
    conn
)


**Exercise 2 (Derived Table):** Using a subquery in the `FROM` clause (derived table), find the course title and its average score. Show only courses where the average active score is above 80. Do not use a CTE.


In [ ]:
# Exercise 2 -- your solution here
# YOUR CODE HERE


**Exercise 3 (Window Function):** Add a `rank_in_course` column -- rank each student within their course by score (1 = highest). Also add a `percentile` column using `PERCENT_RANK() OVER (...)`.


In [ ]:
# Exercise 3 -- your solution here
# YOUR CODE HERE


**Exercise 4 (EXCEPT):** Find all students enrolled in at least one CS course but NOT enrolled in any AI course. Show their name and dept.


In [ ]:
# Exercise 4 -- your solution here
# YOUR CODE HERE


### Git Exercise

**Exercise 5 (Full Workflow):** In your terminal:

1. Create branch `feature/reporting`
2. Add a file `report.py` with a function `summary_report(conn)` that returns a DataFrame of course averages
3. Commit with a Conventional Commit message
4. Rebase your branch onto `main`
5. Fast-forward merge into `main`
6. Push to GitHub
7. Run `git log --oneline --graph` and paste the output below


In [ ]:
# Paste your git log output here as a comment:
# 


---
## Week 3 Summary

### SQL Skills Acquired

| Topic | Key Syntax | When to Use |
|---|---|---|
| Scalar Subquery | `WHERE col > (SELECT AVG(col) FROM t)` | Compare to a single computed value |
| Table Subquery (IN) | `WHERE id IN (SELECT id FROM t WHERE ...)` | Filter by membership in a set |
| Derived Table | `FROM (SELECT ... GROUP BY ...) AS alias` | Aggregate before joining |
| Correlated Subquery | `(SELECT COUNT(*) FROM t WHERE t.id = outer.id)` | Per-row calculations |
| EXISTS | `WHERE EXISTS (SELECT 1 FROM t WHERE ...)` | Check for existence of related rows |
| CTE (Basic) | `WITH name AS (...) SELECT ... FROM name` | Name and reuse a subquery |
| Chained CTEs | `WITH a AS (...), b AS (...SELECT ... FROM a...)` | Multi-step analysis |
| ROW_NUMBER | `ROW_NUMBER() OVER (PARTITION BY x ORDER BY y)` | Ranking within groups |
| RANK / DENSE_RANK | `RANK() OVER (...)` | Ranking with tie handling |
| LAG / LEAD | `LAG(col, n) OVER (ORDER BY ...)` | Compare to previous/next row |
| Running SUM | `SUM(col) OVER (ORDER BY ... ROWS UNBOUNDED PRECEDING)` | Cumulative totals |
| UNION / UNION ALL | `SELECT ... UNION SELECT ...` | Combine result sets |
| INTERSECT | `SELECT ... INTERSECT SELECT ...` | Common rows only |
| EXCEPT | `SELECT ... EXCEPT SELECT ...` | Rows in A not in B |
| Index | `CREATE INDEX idx ON table(col)` | Speed up WHERE / JOIN |

### Git Skills Acquired

| Concept | Command(s) | Key Point |
|---|---|---|
| Configure identity | `git config --global user.name` | Done once per machine |
| Initialise repo | `git init` | Creates `.git` folder |
| Stage changes | `git add` | Moves to index/staging area |
| Commit snapshot | `git commit -m` | Use Conventional Commits |
| View history | `git log --oneline --graph` | Add `--all` to see all branches |
| Create branch | `git checkout -b name` | Isolated development |
| Merge branch | `git merge --no-ff` | `--no-ff` preserves branch history |
| Resolve conflict | Edit file -> `git add` -> `git commit` | Remove conflict markers |
| Connect remote | `git remote add origin <url>` | Points to GitHub |
| Push changes | `git push -u origin main` | `-u` sets upstream |
| Rebase | `git rebase main` | Replay commits on top of main |
| Interactive rebase | `git rebase -i HEAD~n` | Squash, reword, drop commits |
| Pull Request | GitHub UI | Code review before merging |

---

**You are now ready for Week 4 -- ML Prerequisites (NumPy, Pandas, Matplotlib)!**


In [ ]:
# Cleanup
import os
os.chdir("/")
try:
    conn.close()
except Exception:
    pass
print("All done -- school.db closed.")
